# Phase 3: Rhetorical Roles - Knowledge Distillation into the Encoder-Decoder

## Task
Rhetorical role labelling: every sentence of a legal document (Corporate Law and IT domains) gets one of 13 rhetorical-role labels (facts, arguments, ruling and so on). Scored with macro F1 over the labels.

Model: the student is the pre-trained encoder-decoder with frozen weights, LoRMA on the encoder and a new sentence head. The new head learns faster than the LoRMA matrices, and the small LoRMA correction is computed in 32-bit for stability. The student learns from the gold labels and from the teacher's saved probabilities. The loss mixes the normal loss on the gold labels with a term that pulls the student's probabilities towards the teacher's; a batch with an invalid loss is skipped.

## Input
- Phase-3 data (needs the HF_TOKEN secret).
- Phase-1 files legal_encdec.pt and tok_encdec.json.
- Teacher probability files (.npz) from the teacher run, found in /kaggle/input by name.

## Flow of the notebook (from input to output)
1. Write the shared helpers: data access, encoder-decoder model, training loop (warm-up, cosine learning rate, mixed precision, early stopping when the dev score stops improving) and submission writer.
2. Set the settings, load the student model, freeze it and add LoRMA to the encoder.
3. Load the sentences and labels of both domains and the teacher probabilities; build each input as a sentence with its neighbouring sentences.
4. Train the student on gold labels and teacher probabilities, then check the dev macro F1 overall and for each domain, the F1 of each label and the most common confusions.
5. Predict one label per test sentence, check that only official labels are used, save the probabilities and write the submission zip.

## Output
- Submission zip with one label per test sentence.
- Saved dev and test probabilities and one zip with all outputs.

In [1]:
# Shared code: data, encoder-decoder model, training loop
import os, glob, json, math, time, random, zipfile, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from tokenizers import Tokenizer
DEV, PAD, BOS, EOS, SEP = "cuda" if torch.cuda.is_available() else "cpu", 0, 1, 2, 3
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./out"; os.makedirs(OUT, exist_ok=True)
# It takes rel (file path in the dataset) as input, searches for a file and returns its path.
def find(rel): h = glob.glob(f"./**/{rel}", recursive=True) + glob.glob(f"/kaggle/input/**/{rel}", recursive=True); return h[0] if h else None
# It takes rel (file path in the dataset) as input, gets a dataset file: local copy if present, otherwise download it and returns a local file path.
def getf(rel):
    from huggingface_hub import hf_hub_download
    try: from kaggle_secrets import UserSecretsClient; t = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception: t = os.environ.get("HF_TOKEN", "")
    return find(rel) or hf_hub_download("Exploration-Lab/CS-781-Capstone", rel, repo_type="dataset", token=t)
jload = lambda rel: json.load(open(getf(rel), encoding="utf-8"))
# It takes rel (file path in the dataset) as input, loads a JSON file as a list of records and returns a list of records.
def rec(rel):
    d = jload(rel)
    if isinstance(d, dict) and isinstance(d.get("text"), list): return [d]
    return [{**v, "id": v.get("id", k)} if isinstance(v, dict) else {"id": k, "text": v} for k, v in d.items()] if isinstance(d, dict) else d
class Attn(nn.Module):
    # It takes d (model size) and h (number of heads) as input, creates the attention layer and returns nothing.
    def __init__(s, d, h): super().__init__(); s.h = h; s.q = nn.Linear(d, d); s.kv = nn.Linear(d, 2 * d); s.o = nn.Linear(d, d)
    # It takes x (input tensor), mem (encoder output), mask (attention mask) and causal (use causal mask) as input, runs self-attention or cross-attention and returns output vectors.
    def forward(s, x, mem=None, mask=None, causal=False):
        m = x if mem is None else mem; B, T, D = x.shape; q = s.q(x).view(B, T, s.h, -1).transpose(1, 2); k, v = s.kv(m).view(B, m.size(1), 2, s.h, -1).permute(2, 0, 3, 1, 4)
        return s.o(F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=causal and mask is None).transpose(1, 2).reshape(B, T, D))
class Block(nn.Module):
    # It takes d (model size), h (number of heads), cross (add cross-attention) and p (dropout rate) as input, creates one Transformer block and returns nothing.
    def __init__(s, d, h, cross=False, p=0.1):
        super().__init__(); s.n1, s.sa, s.cross = nn.LayerNorm(d), Attn(d, h), cross
        if cross: s.nx, s.ca = nn.LayerNorm(d), Attn(d, h)
        s.n2, s.ff, s.dp = nn.LayerNorm(d), nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)), nn.Dropout(p)
    # It takes x (input tensor), mem (encoder output), mm (encoder padding mask), causal (use causal mask) and mask (attention mask) as input, runs one Transformer block and returns output vectors.
    def forward(s, x, mem=None, mm=None, causal=False, mask=None):
        x = x + s.dp(s.sa(s.n1(x), mask=mask, causal=causal))
        if s.cross and mem is not None: x = x + s.dp(s.ca(s.nx(x), mem, mm))
        return x + s.dp(s.ff(s.n2(x)))
class EncDec(nn.Module):
    # It takes V (vocabulary size), d (model size), h (number of heads), ne (encoder layers), nd (decoder layers), L (max length) and p (dropout rate) as input, builds the encoder-decoder model and returns nothing.
    def __init__(s, V=16000, d=640, h=10, ne=6, nd=8, L=1024, p=0.1):
        super().__init__(); s.emb, s.pos = nn.Embedding(V, d), nn.Embedding(L, d); s.enc = nn.ModuleList(Block(d, h, p=p) for _ in range(ne)); s.dec = nn.ModuleList(Block(d, h, True, p) for _ in range(nd)); s.ne_, s.nd_ = nn.LayerNorm(d), nn.LayerNorm(d)
    # It takes x (input tensor) as input, turns token ids into vectors with positions and returns embedding vectors.
    def embed(s, x): return s.emb(x) + s.pos(torch.arange(x.size(1), device=x.device))
    # It takes src (source token ids) as input, runs the encoder over the input text and returns (encoder output, padding mask).
    def encode(s, src):
        mm = (src != PAD)[:, None, None, :]; x = s.embed(src)
        for b in s.enc: x = b(x, mask=mm)
        return s.ne_(x), mm
    # It takes tgt (target token ids), mem (encoder output) and mm (encoder padding mask) as input, runs the decoder and returns next-token scores.
    def decode(s, tgt, mem=None, mm=None):
        x = s.embed(tgt)
        for b in s.dec: x = b(x, mem, mm, causal=True)
        return F.linear(s.nd_(x), s.emb.weight)
    # It takes src (source token ids) and tgt (target token ids) as input, runs encoder then decoder and returns next-token scores (logits).
    def forward(s, src, tgt): return s.decode(tgt, *s.encode(src))
# It loads the pre-trained model, its settings and tokenizer and returns (model, settings, tokenizer).
def base():
    ck = torch.load(find("legal_encdec.pt"), map_location="cpu"); m = EncDec(**ck["cfg"]); m.load_state_dict(ck["sd"]); return m, ck["cfg"], Tokenizer.from_file(find("tok_encdec.json"))
# It takes rows (token lists) and T (length to pad to) as input, pads token lists to the same length and returns a padded tensor of token ids.
def padto(rows, T=None):
    T = T or max(len(r) for r in rows); a = np.full((len(rows), T), PAD, np.int64)
    for i, r in enumerate(rows): a[i, :len(r)] = r[:T]
    return torch.from_numpy(a)
# It takes m (model), ids (token ids) and w (token weights) as input, averages the encoder outputs over the real tokens and returns one vector per row.
def pool(m, ids, w=None):
    h, mm = m.encode(ids); w = (mm[:, 0, 0, :] if w is None else w).to(h.dtype)[..., None]; return (h * w).sum(1) / w.sum(1).clamp(min=1)
# It takes m (model), loss_fn (loss function), sample (batch function), steps (number of steps), lr (learning rate), ev (evaluation function, lower is better), every (how often to evaluate), hours (time limit in hours), acc (gradient accumulation steps), hm (learning-rate multiplier for the new head), warm (warm-up steps), log (how often to print the loss), fill (stretch training to use all the time) and patience (evaluations without improvement before stopping early) as input, trains the model with warm-up, cosine learning rate and evaluation checks, keeps the best weights and returns the model with its best weights restored.
def fit(m, loss_fn, sample, steps, lr, ev, every, hours, acc=1, hm=1, warm=100, log=50, fill=False, patience=0):
    m.to(DEV).train(); ps = [p for p in m.parameters() if p.requires_grad]
    head = [p for n, p in m.named_parameters() if n.startswith("head")]; rest = [p for p in ps if all(p is not h for h in head)]
    opt = torch.optim.AdamW([dict(params=g, mult=k) for g, k in ((rest, 1), (head, hm)) if g], lr=lr, weight_decay=0.01, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda"); t0, best, best_state, run, s, bad = time.time(), 1e9, None, 0.0, 0, 0
    while s < steps:
        if s == 25: budget = max(30, int(hours * 3600 / ((time.time() - t0) / 25))); steps = budget if fill else min(steps, budget)
        if s == 25: t1 = time.time()
        if fill and s == 300: steps = s + int((hours * 3600 - (time.time() - t0)) / ((time.time() - t1) / 275))
        decay = 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * s / steps))
        for g in opt.param_groups: g["lr"] = lr * g["mult"] * min(1, (s + 1) / warm) * decay
        for _ in range(acc):
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"): loss = loss_fn(m, *sample()) / acc
            scaler.scale(loss).backward(); run += loss.item()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(ps, 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); s += 1
        if s % log == 0: print(f"step {s} loss {run / log:.4f}", flush=True); run = 0.0
        if s % every == 0 or s == steps:
            v = ev(m); m.train(); print(f"   eval {v:.4f}", flush=True)
            bad = 0 if v < best else bad + 1
            if v < best: best, best_state = v, {k: p.detach().cpu().clone() for k, p in m.named_parameters() if p.requires_grad}
            if patience and bad >= patience: break
    if best_state: m.load_state_dict(best_state, strict=False)
    return m.eval()
# It takes obj (object) and tag (label) as input, writes predictions into a submission zip and returns a path of the submission zip.
def submit(obj, tag):
    p = f"{OUT}/{tag}_prediction.json"; json.dump(obj, open(p, "w"), ensure_ascii=False); z = f"{OUT}/{tag}_submission.zip"
    with zipfile.ZipFile(z, "w", zipfile.ZIP_DEFLATED) as f: f.write(p, "prediction.json")
    return z
# It takes name as input, zips all output files into one archive and returns a path of the zip file.
def zip_all(name="ALL_outputs.zip"):
    with zipfile.ZipFile(f"{OUT}/{name}", "w", zipfile.ZIP_DEFLATED) as f:
        for p in sorted(glob.glob(f"{OUT}/*")):
            if os.path.isfile(p) and not p.endswith(name) and not p.endswith(".npy"): f.write(p, os.path.basename(p))
    print(f"{OUT}/{name}", f"{os.path.getsize(f'{OUT}/{name}') / 1e6:.1f} MB")

In [2]:
# Settings, load student model and add LoRMA
from sklearn.metrics import f1_score
SMOKE, CTX = False, 1
BS, EPOCHS, HOURS, SENT, SIDE, LR, R, HM = 32, 4, 2.6, 128, 48, 2e-4, 16, 5
ALPHA, TEMP, FRAC, USE_TEST = 0.5, 2.0, 0.3, True
if SMOKE: BS, EPOCHS, SENT, SIDE = 8, 1, 48, 16
m, cfg, tok = base(); tb = lambda xs: [e.ids for e in tok.encode_batch(xs)]
class LoRMA(nn.Module):
    # It takes lin (linear layer) and r (rank) as input, creates the small LoRMA matrices around a linear layer and returns nothing.
    def __init__(s, lin, r): super().__init__(); s.base = lin; s.A = nn.Parameter(torch.randn(r, lin.out_features) / math.sqrt(r)); s.B = nn.Parameter(torch.zeros(lin.out_features, r))
    # It takes x (input tensor) as input, runs the linear layer with the LoRMA update and returns the output of the updated layer.
    def forward(s, x): y = s.base(x); return y + (((y.float() @ s.A.t()) @ s.B.t())).to(y.dtype)
# It takes mod (block) and r (rank) as input, adds LoRMA to every linear layer of a block and returns nothing.
def lorma(mod, r):
    for n, c in mod.named_children(): setattr(mod, n, LoRMA(c, r)) if isinstance(c, nn.Linear) else lorma(c, r)
for p in m.parameters(): p.requires_grad_(False)
for b in m.enc: lorma(b, R)

In [3]:
# Load sentences, labels and teacher predictions
vocab = jload("Phase_3_RR/Train/label_vocab.json"); LAB = [k for k, _ in sorted(vocab.items(), key=lambda kv: kv[1])] if isinstance(vocab, dict) and all(isinstance(x, int) for x in vocab.values()) else list(vocab.values() if isinstance(vocab, dict) else vocab)
LAB = ["None" if x is None else str(x) for x in LAB]; L2I = {l: i for i, l in enumerate(LAB)}      ; K = len(LAB)
# It takes rel (file path in the dataset) and dom (domain name) as input, tokenizes the sentences of each document and returns a list of tokenized documents.
def prep(rel, dom):
    return [dict(id=str(d["id"]), key=f"{dom}|{d['id']}", t=tb([f"{dom}: {s}" for s in d["text"]]), y=[L2I["None" if l is None else str(l)] for l in d.get("labels") or []], q=None) for d in rec(rel)]
tr = prep("Phase_3_RR/Train/CL_train.json", "CL") + prep("Phase_3_RR/Train/IT_train.json", "IT")
dv = prep("Phase_3_RR/Train/CL_dev.json", "CL") + prep("Phase_3_RR/Train/IT_dev.json", "IT")
te = prep("Phase_3_RR/Test/CL_test.json", "CL") + prep("Phase_3_RR/Test/IT_test.json", "IT")
assert len({d["id"] for d in te}) == len(te), "CL and IT test documents share ids - check the official id format"
# It takes pat (file pattern) as input, loads the teacher's saved probabilities and returns teacher probabilities per document.
def tload(pat):
    f = find(pat); z = np.load(f, allow_pickle=True) if f else None
    return {str(k): z[f"p{i}"].astype(np.float32) for i, k in enumerate(z["keys"])} if f else {}
TD, TT, TS = tload("rr1_lt_lorma_dev_avg_dev_probs.npz"), tload("rr1_lt_lorma_dev_avg_test_probs.npz"), tload("*_train_soft.npz")
sj = find("rr1_lt_lorma_summary.json")
if sj: assert json.load(open(sj))["labels"] == LAB, "teacher and student label order differ"
for ds, T_, nm in ((tr, TS, "train"), (dv, TD, "dev"), (te, TT if USE_TEST else {}, "test")):
    for d in ds:
        if d["key"] in T_: assert len(T_[d["key"]]) == len(d["t"]), f"teacher rows != sentences in {d['key']}"; d["q"] = T_[d["key"]]
    print(nm, "documents with teacher probabilities:", sum(d["q"] is not None for d in ds), "/", len(ds))
dv_teacher = [d for d in dv if d["q"] is not None]
for d in dv: d["qd"], d["q"] = d["q"], None
# It takes d (one document) and i (sentence index) as input, builds one input: sentence with its neighbouring sentences and returns (token ids, pooling weights).
def ex(d, i):
    L = sum(d["t"][max(0, i - CTX):i], [])[-SIDE:] if CTX else []; R_ = sum(d["t"][i + 1:i + 1 + CTX], [])[:SIDE] if CTX else []; s = d["t"][i][:SENT]
    return L + [SEP] + s + [SEP] + R_, [0] * (len(L) + 1) + [1] * len(s) + [0] * (len(R_) + 1)
# It takes ds (documents) and idx ((document, sentence) pairs) as input, builds a padded batch of sentences and returns padded tensors for the batch.
def batch(ds, idx): e = [ex(ds[a], b) for a, b in idx]; return padto([x[0] for x in e]).to(DEV), padto([x[1] for x in e]).to(DEV)
gold = [(a, i) for a, d in enumerate(tr) for i in range(len(d["t"]))]; soft = [(a, i) for a, d in enumerate(te) if d["q"] is not None for i in range(len(d["t"]))]
cnt = np.bincount([y for d in tr for y in d["y"]], minlength=K); wt = torch.tensor((cnt.sum() / (cnt + 1.0)) ** 0.5, dtype=torch.float32); wt /= wt.mean()
print(len(gold), "gold sentences |", len(soft), "teacher-only sentences | class counts", cnt.tolist())

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

train documents with teacher probabilities: 80 / 80
dev documents with teacher probabilities: 10 / 10
test documents with teacher probabilities: 10 / 10
16845 gold sentences | 2197 teacher-only sentences | class counts [4098, 107, 1613, 1348, 1099, 284, 1945, 569, 3, 874, 4301, 307, 297]


In [4]:
# Train the student with distillation, check dev score and per-label results
m.head = nn.Linear(cfg["d"], K); rng = np.random.default_rng(0)
# It picks a random batch of training examples and returns one training batch.
def sample():
    ns = int(BS * FRAC) if soft else 0; g = [(0, gold[i]) for i in rng.integers(0, len(gold), BS - ns)] + [(1, soft[i]) for i in rng.integers(0, len(soft), ns)]
    xs, ws = zip(*[ex((tr, te)[s][a], b) for s, (a, b) in g]); y, q = [], []
    for s, (a, b) in g:
        d = (tr, te)[s][a]; y.append(d["y"][b] if s == 0 else -1); q.append(d["q"][b] if d["q"] is not None else np.zeros(K, np.float32))
    return padto(xs).to(DEV), padto(ws).to(DEV), torch.tensor(y).to(DEV), torch.tensor(np.array(q)).to(DEV)
# It takes m (model), x (input ids), w (pooling weights), y (labels) and q (teacher probabilities) as input, computes the training loss for one batch and returns a loss value.
def lossf(m, x, w, y, q):
    lg = m.head(pool(m, x, w)).float(); g, h = y >= 0, q.sum(-1) > 0
    ce = (F.cross_entropy(lg, y.clamp(min=0), weight=wt.to(DEV), reduction="none") * g).sum() / g.sum().clamp(min=1)
    tq = q.clamp_min(1e-9) ** (1 / TEMP); tq = tq / tq.sum(-1, keepdim=True)
    kd = ((tq * (tq.log() - F.log_softmax(lg / TEMP, -1))).sum(-1) * h).sum() / h.sum().clamp(min=1)
    loss = (1 - ALPHA) * ce + ALPHA * TEMP ** 2 * kd
    return loss if torch.isfinite(loss) else lg.sum() * 0
# It takes ds (documents) as input, gets the student's probabilities for every document and returns probabilities per document.
@torch.no_grad()
def probs(ds):
    m.eval(); idx = [(a, i) for a, d in enumerate(ds) for i in range(len(d["t"]))]; out = []
    for j in range(0, len(idx), 64):
        x, w = batch(ds, idx[j:j + 64])
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEV == "cuda"): out.append(F.softmax(m.head(pool(m, x, w)).float(), -1).cpu().numpy())
    P_ = np.concatenate(out); r, c = {}, 0
    for d in ds: r[d["key"]] = P_[c:c + len(d["t"])]; c += len(d["t"])
    return r
mf1 = lambda ds, P_: f1_score([y for d in ds for y in d["y"]], np.concatenate([P_[d["key"]] for d in ds]).argmax(-1), average="macro")
sub = dv[:max(1, len(dv) // 4)]; steps = max(10, EPOCHS * (len(gold) + len(soft)) // BS)
print("trainable / total parameters:", sum(p.numel() for p in m.parameters() if p.requires_grad), "/", sum(p.numel() for p in m.parameters()))
fit(m, lossf, sample, steps, LR, lambda m: -mf1(sub, probs(sub)), 500 if not SMOKE else 20, HOURS, hm=HM, fill=not SMOKE)
Pd = probs(dv); f1 = mf1(dv, Pd); print("student (LoRMA + KD) dev macro-F1:", round(f1, 4))
yd_all = [y for d in dv for y in d["y"]]; pd_all = np.concatenate([Pd[d["key"]] for d in dv]).argmax(-1)
for dom in ("CL", "IT"):
    ds = [d for d in dv if d["key"].startswith(dom + "|")]; print(f"student dev macro-F1 {dom}:", round(mf1(ds, Pd), 4), "|", sum(len(d["t"]) for d in ds), "sentences")
from sklearn.metrics import precision_recall_fscore_support as prf, confusion_matrix
p_, r_, f_, n_ = prf(yd_all, pd_all, labels=list(range(K)), zero_division=0); print("per label (which roles stay difficult):")
for i in np.argsort(f_): print(f"   {LAB[i]:24s} F1 {f_[i]:.3f}  P {p_[i]:.3f}  R {r_[i]:.3f}  support {n_[i]}")
cm = confusion_matrix(yd_all, pd_all, labels=list(range(K))); np.fill_diagonal(cm, 0); print("most frequent confusions (gold -> predicted):", [(LAB[a], LAB[b], int(cm[a, b])) for a, b in zip(*np.unravel_index(np.argsort(-cm, axis=None)[:5], cm.shape))])
if dv_teacher:
    T_ = {d["key"]: d["qd"] for d in dv_teacher}; print("teacher dev macro-F1:", round(mf1(dv_teacher, T_), 4), "| student on the same documents:", round(mf1(dv_teacher, Pd), 4),
        "| average of both:", round(mf1(dv_teacher, {k: (T_[k] + Pd[k]) / 2 for k in T_}), 4))

trainable / total parameters: 1169965 / 103084141
step 50 loss 1.6694
step 100 loss 1.5455
step 200 loss 1.2430
step 250 loss 1.2958
step 400 loss 1.2653
step 450 loss 1.2065
step 500 loss 1.1145
   eval -0.2160
step 550 loss 1.1504
step 600 loss 1.1349
step 650 loss 1.1369
step 700 loss 1.1609
step 750 loss 1.1095
step 800 loss 1.1001
   eval -0.2361
student (LoRMA + KD) dev macro-F1: 0.2162
student dev macro-F1 CL: 0.2373 | 1396 sentences
student dev macro-F1 IT: 0.2219 | 746 sentences
per label (which roles stay difficult):
   Issue                    F1 0.000  P 0.000  R 0.000  support 15
   Dissent                  F1 0.000  P 0.000  R 0.000  support 0
   PrecedentOverruled       F1 0.000  P 0.000  R 0.000  support 0
   RulingByLowerCourt       F1 0.036  P 0.250  R 0.020  support 102
   None                     F1 0.051  P 0.167  R 0.030  support 33
   PrecedentNotReliedUpon   F1 0.058  P 0.105  R 0.040  support 50
   ArgumentRespondent       F1 0.089  P 0.182  R 0.059  support 27

In [5]:
# Predict on test set and make submission zip
Pt = probs(te); pred = {d["id"]: [LAB[x] for x in Pt[d["key"]].argmax(-1)] for d in te}
assert all(len(v) == len(d["t"]) for d, v in zip(te, pred.values()))
for nm, P_ in (("dev", Pd), ("test", Pt)): ks = sorted(P_); np.savez_compressed(f"{OUT}/p3_kd_lorma_{nm}_probs.npz", keys=np.array(ks), **{f"p{i}": P_[k].astype(np.float16) for i, k in enumerate(ks)})
assert all(l in LAB for v in pred.values() for l in v)
print(submit(pred, f"p3_kd_lorma_dev{f1:.4f}"), len(pred), "documents"); zip_all()

/kaggle/working/p3_kd_lorma_dev0.2162_submission.zip 10 documents
/kaggle/working/ALL_outputs.zip 0.1 MB
